# PTBR-Reranker — Phase 3: training (Kaggle, GPU free tier)

Valida o treino end-to-end do cross-encoder em CPU/GPU Kaggle free, conectando Phases 1+2+3:

1. Clona o repositório `tardellirs/ptbr-reranker`.
2. Instala dependências.
3. Roda `scripts/reproduce.sh --small --skip-eval` — download mMARCO + qrels sintéticos + mining Serafim + build_triples + treino curto (5 steps).
4. Inspeciona o checkpoint salvo em `runs/hardneg/best/` + `training_config.json`.
5. Roda inferência qualitativa com o checkpoint reciém-treinado.

Esse é o validador de **code path** end-to-end. Mesmo com 5 steps em 100 triples sintéticos, prova que o pipeline conecta corretamente do download até o checkpoint utilizável.

**Treino real para o paper** acontece em Runpod A100 SXM com `--full` (sem `--small`), ~30–40h.

## 1. Clonar repositório e instalar dependências

In [ ]:
!rm -rf /kaggle/working/ptbr-reranker
!git clone --depth 1 https://github.com/tardellirs/ptbr-reranker.git /kaggle/working/ptbr-reranker
%cd /kaggle/working/ptbr-reranker
!git log --oneline -1

In [ ]:
!pip install -q -e ".[dev]" 2>&1 | tail -5

In [ ]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Pipeline completo em modo `--small` via `scripts/reproduce.sh`

Um único comando faz download → sintéticos qrels → mining → build_triples → treino curto.

In [ ]:
!bash scripts/reproduce.sh --small --skip-eval 2>&1 | tail -50

## 3. Inspecionar checkpoint salvo e config persistido

In [ ]:
import json
from pathlib import Path

best_dir = Path("runs/hardneg/best")
print("Files in runs/hardneg/best/:")
for p in sorted(best_dir.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f"  {p.relative_to(best_dir)} ({size_mb:.1f} MB)")

cfg_path = Path("runs/hardneg/training_config.json")
if cfg_path.exists():
    print("\nResolved training config:")
    print(json.dumps(json.loads(cfg_path.read_text()), indent=2))

## 4. Inferência qualitativa com o checkpoint reciém-treinado

O modelo foi treinado por apenas 5 steps em 100 triples sintéticos — os scores não refletem relevância real. Aqui apenas validamos que o checkpoint é carregável e produz scores.

In [ ]:
import os
from pathlib import Path

import torch
from sentence_transformers import CrossEncoder

best_dir = Path("runs/hardneg/best").resolve()
if not best_dir.exists() or not any(best_dir.iterdir()):
    print(f"Best checkpoint not found at {best_dir} - training likely did not complete.")
    print("Falling back to base Albertina-100m to demonstrate the inference code path.")
    checkpoint = "PORTULAN/albertina-100m-portuguese-ptbr-encoder"
else:
    checkpoint = str(best_dir)
print(f"Loading: {checkpoint}")

os.environ.setdefault("WANDB_DISABLED", "true")
model = CrossEncoder(
    checkpoint,
    max_length=128,
    num_labels=1,
    model_kwargs={"torch_dtype": torch.float32},
)
model.model.float()

query = "qual a capital do Brasil?"
passages = [
    "Brasília é a capital federal do Brasil desde 1960.",
    "São Paulo é a maior cidade do Brasil.",
    "Receita de bolo de chocolate com cobertura de brigadeiro.",
]
scores = model.predict([(query, p) for p in passages])
for passage, score in sorted(zip(passages, scores.tolist(), strict=True), key=lambda x: -x[1]):
    print(f"{score:+.4f}  {passage}")

## 5. Checklist final

Se todas as células acima passaram:

- [x] `scripts/reproduce.sh --small` rodou end-to-end (download → mining → build_triples → train).
- [x] Checkpoint salvo em `runs/hardneg/best/`.
- [x] `training_config.json` persistido (artefato de reprodutibilidade).
- [x] Checkpoint carregável via `CrossEncoder()`.
- [x] Inferência produz scores numéricos.

Phase 3 (code path) validada. Para o treino real do paper:

1. Migrar para Runpod A100 SXM Community ($1.39/h).
2. Rodar `scripts/runpod_train.sh` (configura ambiente + `scripts/reproduce.sh --config configs/train_hardneg.yaml`).
3. ~30–40h, custo total ~$42–55.

Registrar resultado em `docs/lab_notebook.md`.